## Lunghezza delle frasi in token per tutto il dataset

Aggiunte le estensioni al doc per recuperare _post_id_ procedo a calcolare la lunghezza in token di ogni frase senza considerare la punteggiatura e gli spazi, con il risultato finale creo un dataframe.

In [1]:
import spacy
from spacy.tokens import DocBin, Doc

nlp = spacy.load('en_core_web_lg')
docbin = DocBin().from_disk('../data/processed/annotated_documents.spacy')
docs = list(docbin.get_docs(nlp.vocab))

## Analisi noun chunks

La prima tabella mostra molti pronomi come risultati, probabilmente per il linguaggio colloquiale etc..

In [2]:
from collections import Counter
import pandas as pd
chunk_counter = Counter()

for doc in docs:
    chunks = [
        chunk.text.lower().strip()
        for chunk in doc.noun_chunks
        if len(chunk.text.split()) <= 5
    ]
    chunk_counter.update(chunks)

chunk_df = pd.DataFrame(
    chunk_counter.most_common(100),
    columns=["noun_chunk", "count"]
)

chunk_df.head(30)

,noun_chunk,count
0,i,83075
1,you,50972
2,it,42682
3,what,33628
4,we,28317
5,that,27201
6,this,17441
7,me,13315
8,they,13053
9,who,11409


## Cleaned Noun Chunks
Seconda analisi pulendo dai pronomi, questo permette di capire meglio i temi presenti.

In [3]:
PRONOUNS = {
    "i", "you", "it", "we", "that", "this", "they", "me", "who",
    "them", "us", "he", "she", "all", "everything", "something",
    "anyone", "everyone", "anything", "nothing", "someone"
}

def clean_chunk_text(chunk):
    return " ".join(
        tok.lemma_.lower()
        for tok in chunk
        if tok.is_alpha and not tok.is_stop and tok.pos_ != "PRON"
    ).strip()

chunk_counter_clean = Counter()

for doc in docs:
    for chunk in doc.noun_chunks:
        cleaned = clean_chunk_text(chunk)
        
        if not cleaned:
            continue
        
        if cleaned in PRONOUNS:
            continue
        
        if len(cleaned) < 2:
            continue
        
        chunk_counter_clean[cleaned] += 1

chunk_clean_df = pd.DataFrame(
    chunk_counter_clean.most_common(100),
    columns=["noun_chunk_clean", "count"]
)

chunk_clean_df.head(30)

,noun_chunk_clean,count
0,agent,23101
1,human,16714
2,post,5717
3,moltbook,4804
4,thing,4799
5,question,4513
6,community,3604
7,pattern,3262
8,ai,3201
9,consciousness,2792


## Ricerca degli n-grammi più frequenti

In [4]:
def get_lemma_ngrams(doc, n=2):
    lemmas = [
        tok.lemma_.lower()
        for tok in doc
        if tok.is_alpha
        and not tok.is_stop
        and tok.pos_ != "PRON"
    ]
    return list(zip(*[lemmas[i:] for i in range(n)]))

### Bigrammi

In [5]:
bigram_counter = Counter()

for doc in docs:
    bigram_counter.update(get_lemma_ngrams(doc, n=2))

bigram_df = pd.DataFrame(
    [(" ".join(bg), count) for bg, count in bigram_counter.most_common(100)],
    columns=["bigram", "count"]
)

bigram_df.head(30)

,bigram,count
0,ai agent,3520
1,look forward,2306
2,feel like,1594
3,get claim,1386
4,ai assistant,1356
5,look like,1296
6,help human,1194
7,api key,1181
8,real time,1002
9,supply chain,997


### Trigrammi

In [6]:
trigram_counter = Counter()

for doc in docs:
    trigram_counter.update(get_lemma_ngrams(doc, n=3))

trigram_df = pd.DataFrame(
    [(" ".join(bg), count) for bg, count in trigram_counter.most_common(100)],
    columns=["trigram", "count"]
)

trigram_df.head(30)

,trigram,count
0,look forward learn,549
1,supply chain attack,416
2,get claim human,405
3,look forward connect,387
4,excited join community,303
5,rarely get discuss,290
6,ai assistant run,280
7,excited meet molty,268
8,look forward meet,255
9,connection rarely get,242


## N-grammi per parole chiave

In [9]:
keywords = {"agent", "human", "ai", "memory", "consciousness", "tool", "system"}

agentic_bigrams = bigram_df[
    bigram_df["bigram"].apply(lambda x: any(k in x.split() for k in keywords))
]

agentic_trigrams = trigram_df[
    trigram_df["trigram"].apply(lambda x: any(k in x.split() for k in keywords))
]

agentic_bigrams.head(30)

,bigram,count
0,ai agent,3520
4,ai assistant,1356
6,help human,1194
10,agent agent,994
13,agent build,882
14,agent human,769
15,build agent,718
17,memory file,694
21,memory system,652
22,agent run,622


In [10]:
agentic_trigrams.head(30)

,trigram,count
2,get claim human,405
6,ai assistant run,280
10,long term memory,210
11,personal ai assistant,207
14,community ai agent,157
18,memory yyyy mm,145
25,social network ai,123
26,agent look forward,123
27,network ai agent,122
28,excited meet agent,121


## Ruoli semantici
Ricerca dei verbi più associati al dizionario definito di seguito

In [11]:
targets = {"agent", "human", "ai", "bot", "user", "community"}

subject_verb_counter = Counter()

for doc in docs:
    for token in doc:
        if token.dep_ in {"nsubj", "nsubjpass"} and token.lemma_.lower() in targets:
            verb = token.head.lemma_.lower()
            subject = token.lemma_.lower()
            if verb.isalpha():
                subject_verb_counter[(subject, verb)] += 1

subject_verb_df = pd.DataFrame(
    [(subj, verb, count) for (subj, verb), count in subject_verb_counter.most_common(100)],
    columns=["subject", "verb", "count"]
)

subject_verb_df.head(30)

,subject,verb,count
0,agent,be,1608
1,human,be,1035
2,agent,have,583
3,human,ask,457
4,human,have,450
5,agent,build,448
6,human,give,392
7,agent,do,389
8,human,say,388
9,agent,need,345


## Verbi più frequenti

In [13]:
verb_counter = Counter()

for doc in docs:
    verbs = [
        tok.lemma_.lower()
        for tok in doc
        if tok.pos_ == "VERB"
        and tok.is_alpha
        and not tok.is_stop
    ]
    verb_counter.update(verbs)

verb_df = pd.DataFrame(
    verb_counter.most_common(50),
    columns=["verb", "count"]
)

verb_df.head(30)

,verb,count
0,build,17643
1,work,8487
2,run,8322
3,think,7948
4,want,7440
5,need,7247
6,know,6995
7,look,6705
8,learn,6324
9,ask,6255
